In [3]:
import json
from pathlib import Path
from modules.tools import normalize
import pandas as pd
from xgboost import XGBClassifier

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import cross_val_score
from sklearn import tree, svm
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import classification_report, roc_curve

/Users/maksi/Develop/ai-llm/ai-detection/.venv/lib/python3.13/site-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


In [4]:
def load_config(config_path):
    with open(config_path, "r") as f:
        return json.load(f)

In [5]:
config = load_config(Path("configs/load_data.json"))

load_data_folder = Path(config["preprocess_folder"]) / "tokenized"
save_folder = Path(config["save_models_folder"])
dataset_name = config["dataset_for_experiment"]
features = config["features"]
models = config["models"]
batch_size = config["batch_size"]
epochs_num = config["epochs_num"]
learning_rate = config["learning_rate"]
device = config["device"]
fp16 = config["fp16"]

In [6]:
def divide_data(dataset, features):

    _, preprocessor = normalize(dataset[dataset["split"] == "train"], features=features)
    dataset, preprocessor = normalize(dataset, features=features, preprocessor=preprocessor)

    columns_y = "label"

    X_train = dataset[dataset["split"] == "train"][features]
    X_val = dataset[dataset["split"] == "validate"][features]
    X_test = dataset[dataset["split"] == "test"][features]

    y_train = dataset[dataset["split"] == "train"][columns_y]
    y_val = dataset[dataset["split"] == "validate"][columns_y]
    y_test = dataset[dataset["split"] == "test"][columns_y]

    return X_train, X_val, X_test, y_train, y_val, y_test, preprocessor

In [24]:
gnb = GaussianNB()
lor = LogisticRegression(max_iter=1000)
knn = KNeighborsClassifier()
raf = RandomForestClassifier()
tr = tree.DecisionTreeClassifier()
cvm = svm.SVC()
xgb = XGBClassifier(max_iter=1000)
etc = ExtraTreesClassifier()
models = {
    "GaussianNB": gnb,
    "LogisticRegression": lor,
    "KNeighbors": knn,
    "RandomForest": raf,
    "DecisionTree": tr,
    # "CVM": cvm,
    "XGBoost": XGBClassifier(tree_method="hist", n_jobs=1),
    "ExtraTrees": ExtraTreesClassifier(),
}

tfidf = TfidfVectorizer(stop_words=None)

dataset_path = Path("ai_detection/data/processed/tokenized/bert-base-uncased_MAGE.parquet")
dataset_df = pd.read_parquet(dataset_path, columns=[*features, "split", "label", "text"])

from scipy.sparse import csr_matrix, hstack

dataset_df = dataset_df.reset_index(drop=True)
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor = divide_data(
    dataset_df, features
)

tfidf = TfidfVectorizer(stop_words=None)
text_train = tfidf.fit_transform(dataset_df.loc[X_train.index, "text"])
text_val = tfidf.transform(dataset_df.loc[X_val.index, "text"])
text_test = tfidf.transform(dataset_df.loc[X_test.index, "text"])

X_train_c = hstack([text_train, csr_matrix(X_train.to_numpy())], format="csr")
X_val_c = hstack([text_val, csr_matrix(X_val.to_numpy())], format="csr")
X_test_c = hstack([text_test, csr_matrix(X_test.to_numpy())], format="csr")

for name, X_tr, X_va in [
    ("TF-IDF", text_train, text_val),
    ("TF-IDF + 20 признаков", X_train_c, X_val_c),
]:
    model = LogisticRegression(solver="lbfgs", max_iter=1000)
    model.fit(X_tr, y_train)
    pred = model.predict(X_va)

    print(f"\n{name} | итераций: {model.n_iter_[0]}")
    print(classification_report(y_val, pred, digits=4, zero_division=0))



TF-IDF | итераций: 102
              precision    recall  f1-score   support

           0     0.7704    0.4553    0.5724     28799
           1     0.6056    0.8604    0.7109     27993

    accuracy                         0.6550     56792
   macro avg     0.6880    0.6579    0.6416     56792
weighted avg     0.6892    0.6550    0.6406     56792


TF-IDF + 20 признаков | итераций: 515
              precision    recall  f1-score   support

           0     0.8075    0.5369    0.6450     28799
           1     0.6457    0.8683    0.7406     27993

    accuracy                         0.7003     56792
   macro avg     0.7266    0.7026    0.6928     56792
weighted avg     0.7278    0.7003    0.6921     56792



In [22]:




model = LogisticRegression(max_iter=1000)
model.fit(X_train_c, train["label"])
val_predictions = model.predict(X_val_c)

print(classification_report(val["label"], val_predictions))


              precision    recall  f1-score   support

           0       0.75      0.54      0.63     28799
           1       0.63      0.81      0.71     27993

    accuracy                           0.67     56792
   macro avg       0.69      0.68      0.67     56792
weighted avg       0.69      0.67      0.67     56792

